In [ ]:

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("silver_schema", "lab8_dev_silver")
dbutils.widgets.text("gold_schema", "lab8_dev_gold")

catalog = dbutils.widgets.get("catalog")
silver = f"{catalog}.{dbutils.widgets.get('silver_schema')}"
gold = f"{catalog}.{dbutils.widgets.get('gold_schema')}"

# 1. Required tables must exist and contain data
tables = [
    f"{silver}.yield_and_climate",
    f"{silver}.environment",
    f"{silver}.dim_country_scd1",
    f"{gold}.dim_country",
    f"{gold}.dim_crop",
    f"{gold}.dim_year",
    f"{gold}.fact_agri_environment",
    f"{gold}.agg_crop_year",
    f"{gold}.agg_country_year",
]

for table in tables:
    assert spark.catalog.tableExists(table), f"Missing table: {table}"
    assert spark.table(table).limit(1).count() > 0, f"Empty table: {table}"
    print(f"PASS: {table}")

# 2. Fact table must not contain NULL business keys
fact = spark.table(f"{gold}.fact_agri_environment")

null_keys = fact.filter("""
    country_key IS NULL
    OR crop_key IS NULL
    OR year_key IS NULL
""").limit(1).count()

assert null_keys == 0, "Fact table contains NULL business keys"
print("PASS: Fact business keys are NOT NULL")

# 3. Dimension keys must be unique
dimensions = {
    "dim_country": "country_code",
    "dim_crop": "crop_key",
    "dim_year": "year_key",
}

for table_name, key in dimensions.items():
    df = spark.table(f"{gold}.{table_name}")

    duplicate_exists = (
        df.groupBy(key)
        .count()
        .filter("count > 1")
        .limit(1)
        .count()
    )

    assert duplicate_exists == 0, f"Duplicate {key} in {table_name}"
    print(f"PASS: {table_name} keys are unique")

print("ALL DATA QUALITY CHECKS PASSED")
